In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

atom_props = pd.read_excel("atom_props.xlsx")

print(atom_props.shape)
print(atom_props.columns)

In [0]:
import pandas as pd
import numpy as np

COMP_FILE = "composition_dataset_db_ml.csv"
ATOM_FILE = "atom_props.xlsx"
OUTPUT_FILE = "descriptor_dataset_db_ml.csv"

# load data
comp_df = pd.read_csv(COMP_FILE)
atom_df = pd.read_excel(ATOM_FILE)

# index atomic properties by element
atom_df = atom_df.set_index("Element")

# detect element columns
element_cols = [c for c in comp_df.columns if c in atom_df.index]

# atomic properties
prop_cols = atom_df.columns.tolist()

descriptors = []

for _, row in comp_df.iterrows():

    stoich = row[element_cols]
    stoich = stoich[stoich > 0]

    elements = stoich.index
    amounts = stoich.values

    desc = {}

    for prop in prop_cols:

        values = atom_df.loc[elements, prop].values

        # remove NaNs
        mask = ~np.isnan(values)
        values = values[mask]
        amounts_valid = amounts[mask]

        if len(values) == 0:
            continue

        total = amounts_valid.sum()

        wmean = np.sum(values * amounts_valid) / total
        wstd = np.sqrt(np.sum(amounts_valid * (values - wmean)**2) / total)
        prange = values.max() - values.min()

        desc[f"{prop}_mean"] = wmean
        desc[f"{prop}_std"] = wstd
        desc[f"{prop}_range"] = prange

    descriptors.append(desc)

desc_df = pd.DataFrame(descriptors)

final_df = pd.concat([comp_df, desc_df], axis=1)

final_df.to_csv(OUTPUT_FILE, index=False)

print("Descriptors generated.")
print(f"Saved to {OUTPUT_FILE}")

In [0]:
display(final_df)

In [0]:
import pandas as pd

INPUT_FILE = "descriptor_dataset_db_ml.csv"
OUTPUT_FILE = "descriptor_numeric_dataset_db_ml.csv"

df = pd.read_csv(INPUT_FILE)

# keep only numeric columns
df_numeric = df.select_dtypes(include=["number"])

df_numeric.to_csv(OUTPUT_FILE, index=False)

print("Non-numeric columns removed.")
print("New shape:", df_numeric.shape)

In [0]:
display (df_numeric)

In [0]:
import pandas as pd
import numpy as np

INPUT_FILE = "descriptor_dataset_db_ml.csv"
ATOM_FILE = "atom_props.xlsx"
OUTPUT_FILE = "full_descriptors_dataset_db_ml.csv"

df = pd.read_csv(INPUT_FILE)
atom = pd.read_excel(ATOM_FILE).set_index("Element")

# element columns
element_cols = [c for c in df.columns if c in atom.index]

# Shannon-like proxy: use atomic_radius
radius_col = "atomic_radius"

def compute_descriptors(row):
    stoich = row[element_cols]
    stoich = stoich[stoich > 0]

    elements = stoich.index
    amounts = stoich.values

    # remove elements without radius
    radii = atom.loc[elements, radius_col].values
    mask = ~np.isnan(radii)

    elements = elements[mask]
    amounts = amounts[mask]
    radii = radii[mask]

    if len(elements) < 2:
        return pd.Series({"tolerance_factor": np.nan,
                          "octahedral_factor": np.nan})

    # separate oxygen
    is_oxygen = elements == "O"

    if not is_oxygen.any():
        return pd.Series({"tolerance_factor": np.nan,
                          "octahedral_factor": np.nan})

    r_O = radii[is_oxygen][0]

    # cations only
    elements_cat = elements[~is_oxygen]
    amounts_cat = amounts[~is_oxygen]
    radii_cat = radii[~is_oxygen]

    # heuristic split: largest radius = A-site, rest = B-site
    idx_max = np.argmax(radii_cat)

    r_A = radii_cat[idx_max]
    r_B = np.mean(np.delete(radii_cat, idx_max)) if len(radii_cat) > 1 else radii_cat[idx_max]

    # tolerance factor
    t = (r_A + r_O) / (np.sqrt(2) * (r_B + r_O))

    # octahedral factor
    mu = r_B / r_O

    return pd.Series({
        "tolerance_factor": t,
        "octahedral_factor": mu,
        "r_A": r_A,
        "r_B": r_B
    })


struct_df = df.apply(compute_descriptors, axis=1)

full_descriptors_dataset = pd.concat([df, struct_df], axis=1)

full_descriptors_dataset .to_csv(OUTPUT_FILE, index=False)

print("Structural descriptors added.")

In [0]:
display (full_descriptors_dataset)